# Test All Recent Changes

Run each cell in order to verify every fix.

In [ ]:
import sys, json
sys.path.insert(0, r"D:\HighRiskPregnancy_XAI_V2")
PROJECT_DIR = r"D:\HighRiskPregnancy_XAI_V2"

---
## 1. All-Normal Patient (was: would crash or show misleading risks)

In [ ]:
from pipeline import run_complete_pipeline, pipeline_result_to_json

normal_patient = {
    "Age": 25, "G": 1, "P": 0, "L": 0, "A": 0, "D": 0,
    "SystolicBP": 110, "DiastolicBP": 70, "RBS": 100,
    "BodyTemp": 98.6, "HeartRate": 75, "HB": 12,
    "HBA1C": 5.0, "RR": 16
}

result = run_complete_pipeline(normal_patient, PROJECT_DIR)
print("Prediction:", result["prediction"])
print("Low risk %:", result["low_risk_probability"])
print("High risk %:", result["high_risk_probability"])
print("Reports generated:", len(result["reports"]), "(should be 0 for Low Risk)")
print("Primary risk:", result["xai_output"]["primary_risk"]["name"])
print("top_risks empty?", result["top_risks"].empty)

**Expected:**
- Prediction should be "Low Risk" (or possibly "High Risk" depending on model thresholds)
- If Low Risk → reports should be empty (0)
- Primary risk should either be "No significant risk factors identified" or a valid category
- No crash

---
## 2. XAI Output — shows both risk percentages

In [ ]:
print("Prediction output:")
print(json.dumps(result["xai_output"]["prediction"], indent=2))
print()
print("Primary risk:")
print(json.dumps(result["xai_output"]["primary_risk"], indent=2))
print()
print("Disclaimer:")
print(result["xai_output"]["disclaimer"][:100], "...")

**Expected:** Both `high_risk_probability` and `low_risk_probability` should be present.

---
## 3. High-Risk Patient — full pipeline with reports

In [ ]:
high_risk_patient = {
    "Age": 36, "G": 3, "P": 2, "L": 2, "A": 1, "D": 0,
    "SystolicBP": 150, "DiastolicBP": 100, "RBS": 180,
    "BodyTemp": 98.6, "HeartRate": 110, "HB": 7.5,
    "HBA1C": 6.2, "RR": 26
}

result2 = run_complete_pipeline(high_risk_patient, PROJECT_DIR)
print("Prediction:", result2["prediction"])
print("Confidence:", result2["confidence"])
print("Low risk %:", result2["low_risk_probability"])
print("High risk %:", result2["high_risk_probability"])
print()
print("Top risks (should be in rank order, highest→lowest):")
for i, row in result2["top_risks"].iterrows():
    print(f"  {i+1}. {row['Risk Category']} — score: {row['Risk Score']}")

**Expected:**
- Should predict "High Risk"
- Top 3 risks sorted highest→lowest score
- Reports should contain 3 entries in the same order

In [ ]:
print("Number of reports:", len(result2["reports"]))
print()
print("Report order (should match top_risks order above):")
for r in result2["reports"]:
    print(f"  - {r['risk']}: {r['report']['condition_name']}")

**Expected:** Report order should exactly match the top_risks order.

---
## 4. Prompt doesn't say "High Risk" when it's Low Risk

In [ ]:
# Check the prompt template logic
from rag.prompt_builder import build_prompt

prompt = build_prompt("Test Risk", "dummy summary", "dummy context", "Low Risk")

if "classified as High Risk" in prompt:
    print("FAIL: Still says 'High Risk' even when prediction_label='Low Risk'")
elif "classified as Low Risk" in prompt:
    print("PASS: Prompt correctly uses 'Low Risk'")
else:
    print("UNEXPECTED: Neither found")
    print(prompt[200:400])

---
## 5. Boundary values for evaluate_feature

In [ ]:
from xai.clinical_rules import evaluate_feature

print("SystolicBP 139 →", evaluate_feature("SystolicBP", 139)["finding"], "(normal)")
print("SystolicBP 159 →", evaluate_feature("SystolicBP", 159)["finding"], "(moderate)")
print("SystolicBP 160 →", evaluate_feature("SystolicBP", 160)["finding"], "(severe)")

---
## 6. Full JSON output

In [ ]:
json_out = pipeline_result_to_json(result)
print(json.dumps(json_out, indent=2, default=str)[:3000])

---
## All Done

If all cells ran without errors, all fixes are working correctly.